## Koordinatentransformation

Eine Koordinatentransformation ist, allgemein gesprochen, eine Abbildung $\mathbb{R}^n \to \mathbb{R}^n$. In der Regel nutzen wir umkehrbare Transformationen, also bijektive Abbildungen. Besonders angenehm sind **lineare Transformationen**, die sich also durch eine Matrix darstellen lassen. Ist die Matrix invertierbar, so ist die Transformation umkehrbar (rückgängig machbar).

Wir suchen bei der PCA zur Dimensionsreduktion nach einer ganz bestimmten linearen Koordinatentransformation. Daher wollen wir sichergehen, dass unsere Intuition zu linearen Transformationen wenigstens in zwei Dimensionen gut ist.

Hier lassen sich zwei verschiedene Verteilungen samplen (oder ein Bild laden) und der Effekt einer Transformationsmatrix darauf beobachten.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.linalg import svd, sqrtm, inv
from numpy.linalg import matrix_rank, eigvals
from PIL import Image
import io

In [2]:
# Widget-Setup
matrix_inputs = [[widgets.FloatText(value=1 if i == j else 0, description=f"A[{i+1},{j+1}]") for j in range(2)] for i in range(2)]
cov_inputs = [[widgets.FloatText(value=1 if i == j else 0, description=f"Σ[{i+1},{j+1}]") for j in range(2)] for i in range(2)]

dist_type = widgets.Dropdown(options=["Normal", "Uniform"], value="Normal", description="Distribution:")
sample_size = widgets.IntSlider(value=200, min=10, max=1000, step=10, description="Sample size:")
img_upload = widgets.FileUpload(accept='image/*', multiple=False, description="Upload image")

btn_copy_cov = widgets.Button(description="Copy (AAᵀ)⁻¹ to Σ")
btn_copy_cov_inv = widgets.Button(description="Copy √Σ⁻¹ to A (Whiten)")

out = widgets.Output()
ui = widgets.VBox([
    widgets.VBox([widgets.HBox(row) for row in matrix_inputs]),
    widgets.HBox([dist_type, sample_size]),
    widgets.VBox([widgets.HBox(row) for row in cov_inputs]),
    widgets.HBox([btn_copy_cov, btn_copy_cov_inv, img_upload]),
    out
])

In [3]:
# Wesentliche Programmlogik
def parse_matrix(inputs):
    return np.array([[w.value for w in row] for row in inputs])
    
info_box = widgets.HTML()
out_box = widgets.Output()

def format_matrix(name, M):
    return f"<b>{name}</b> = <div><pre>{np.array2string(M, precision=3, suppress_small=True)}</pre>"

def matrix_inline(name, M):
    mat_str = np.array2string(M, precision=3, suppress_small=True, max_line_width=1000)
    return f'<b>{name}</b> = <pre style="display:inline; margin-right: 1em;">{mat_str}</pre>'

def update(*args):
    with out_box:
        clear_output()
        A = parse_matrix(matrix_inputs)
        Σ = parse_matrix(cov_inputs)
        n = sample_size.value
        html = f"<h4>Singulärwertzerlegung</h4>"
        try:
            U, S, VT = svd(A)
            html += "<div><b>A = U · Σ · V<sup>T</sup></b></div>"
            html += '<div style="display:flex; flex-wrap:wrap; gap:1em;">'
            html += matrix_inline("A", A)
            html += matrix_inline("U", U)
            html += matrix_inline("Σ", np.diag(S))
            html += matrix_inline("V<sup>T</sup>", VT)
            html += '</div>'
        except Exception as e:
            html += f"<p style='color:red'>SVD fehlgeschlagen: {e}</p>"

        html += "<h4>Weitere Eigenschaften</h4>"

        is_sym = np.allclose(A, A.T)
        is_spd = is_sym and np.all(eigvals(A) > 0)
        html += f"<p><b>Symmetrisch:</b> {is_sym} &nbsp;&nbsp; <b>Positiv definit:</b> {is_spd}</p>"

        
        html += '<div style="display:flex; flex-wrap:wrap; gap:1em;">'
        A2 = A @ A
        html += matrix_inline("A²", A2)
        
        if is_spd:
            try:
                sqrt_A = sqrtm(A)
                html += matrix_inline("√A", sqrt_A)
            except Exception as e:
                html += f"<p style='color:red'>Matrixwurzel fehlgeschlagen: {e}</p>"

        html += '</div>'
        
        info_box.value = html

        # --- Daten erzeugen + plotten ---
        if dist_type.value == "Normal":
            x = np.random.multivariate_normal([0, 0], Σ, size=n)
        else:
            x = np.random.rand(n, 2) @ Σ.T
        x_trans = x @ A.T

        # Gemeinsame Achsenskalierung
        all_points = np.vstack([x, x_trans])
        xmin, xmax = all_points[:, 0].min(), all_points[:, 0].max()
        ymin, ymax = all_points[:, 1].min(), all_points[:, 1].max()

        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        for a in ax:
            a.set_xlim(xmin, xmax)
            a.set_ylim(ymin, ymax)
            a.set_aspect("equal")
        ax[0].scatter(x[:, 0], x[:, 1], alpha=0.5)
        ax[0].set_title(f"Original Sample X ~ {dist_type.value}")
        ax[1].scatter(x_trans[:, 0], x_trans[:, 1], alpha=0.5, color="orange")
        ax[1].set_title("Transformed Sample AX")
        plt.tight_layout()
        plt.show()

In [4]:
# Button Actions
def copy_cov(*args):
    A = parse_matrix(matrix_inputs)
    Asquared = A @ A
    Asquared_inv = inv(Asquared)
    for i in range(2):
        for j in range(2):
            cov_inputs[i][j].value = Asquared_inv[i, j]

def copy_cov_inv(*args):
    Σ = parse_matrix(cov_inputs)
    if matrix_rank(Σ) == 2:
        Σ_inv = inv(Σ)
        Σ_inv_root = sqrtm(inv(Σ))
        for i in range(2):
            for j in range(2):
                matrix_inputs[i][j].value = Σ_inv_root[i, j]
    else:
        with out:
            print("Covariance matrix is not invertible.")

In [5]:
# Display
for row in matrix_inputs + cov_inputs:
    for w in row:
        w.observe(update, names='value')
dist_type.observe(update, names='value')
sample_size.observe(update, names='value')
img_upload.observe(update, names='value')
btn_copy_cov.on_click(copy_cov)
btn_copy_cov_inv.on_click(copy_cov_inv)

update()
display(widgets.VBox([ui, info_box, out_box]))